# Case Study: Credit Scoring with TuiML

This end-to-end case study demonstrates how to build a **credit scoring** model using the TuiML library. We will use the classic **German Credit** dataset (`credit-g`) to classify loan applicants as **good** or **bad** credit risks.

The workflow covers:
1. **Problem Statement** - Understanding credit risk classification
2. **Data Loading and Exploration** - Inspect the German Credit dataset
3. **Quick Baseline** - One-liner model training
4. **Algorithm Comparison** - Experiment with multiple classifiers
5. **Workflow Approach** - Fluent pipeline with preprocessing
6. **Model Evaluation** - Metrics and confusion matrix
7. **Save and Serve** - Deploy the model as a REST API
8. **Making Predictions** - Call the API with curl and Python
9. **Cleanup** - Stop the server and remove artifacts
10. **Summary and Key Takeaways**

## 1. Problem Statement

**Objective:** Classify loan applicants as **good** or **bad** credit risks based on financial and personal attributes.

**Task Type:** Binary Classification (good vs. bad credit)

**Business Context:**
Credit scoring is a fundamental task in the financial industry. Banks and lending institutions use predictive models to assess the likelihood that a borrower will default on a loan. An accurate credit scoring model helps minimize financial losses from defaults while ensuring creditworthy applicants are not rejected.

**Dataset:** German Credit (`credit-g`)
- **1000 samples** (loan applicants)
- **20 features** (7 numerical, 13 categorical)
- **2 classes:** `good` (700 applicants) and `bad` (300 applicants)
- Collected by Prof. Dr. Hans Hofmann, University of Hamburg

**Key Features:**
| Feature | Type | Description |
|---------|------|-------------|
| `checking_status` | Categorical | Status of existing checking account |
| `duration` | Numeric | Duration of credit in months |
| `credit_history` | Categorical | Past credit repayment behavior |
| `purpose` | Categorical | Purpose of the loan |
| `credit_amount` | Numeric | Credit amount requested |
| `savings_status` | Categorical | Savings account balance |
| `employment` | Categorical | Employment duration |
| `age` | Numeric | Age of applicant |
| `housing` | Categorical | Housing status (rent, own, free) |
| `job` | Categorical | Job qualification level |

**Challenge:** The dataset is **imbalanced** (70% good, 30% bad), which means naive classifiers may simply predict "good" for everyone and achieve 70% accuracy without actually learning meaningful patterns.

## 2. Load and Explore Data

We use TuiML's built-in dataset registry to load the German Credit dataset.

In [1]:
from tuiml.datasets import load_dataset, get_dataset_info

# Get dataset metadata before loading
info = get_dataset_info("credit")
print("Dataset Info:")
for key, value in info.items():
    print(f"  {key}: {value}")

Dataset Info:
  task: classification
  samples: 1000
  features: 20
  classes: 2
  description: German credit risk assessment
  loader: load_credit


In [2]:
# Load the full dataset
dataset = load_dataset("credit-g")
X, y = dataset.X, dataset.y
feature_names = dataset.feature_names

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeature names ({len(feature_names)}):")
for i, name in enumerate(feature_names):
    print(f"  [{i:2d}] {name}")

Feature matrix shape: (1000, 20)
Target vector shape: (1000,)

Feature names (20):
  [ 0] checking_status
  [ 1] duration
  [ 2] credit_history
  [ 3] purpose
  [ 4] credit_amount
  [ 5] savings_status
  [ 6] employment
  [ 7] installment_commitment
  [ 8] personal_status
  [ 9] other_parties
  [10] residence_since
  [11] property_magnitude
  [12] age
  [13] other_payment_plans
  [14] housing
  [15] existing_credits
  [16] job
  [17] num_dependents
  [18] own_telephone
  [19] foreign_worker


In [3]:
import numpy as np

# Explore class distribution
unique, counts = np.unique(y, return_counts=True)
print("Class Distribution:")
for cls, count in zip(unique, counts):
    print(f"  {cls}: {count} ({count / len(y):.1%})")

print(f"\nImbalance Ratio: {counts.max() / counts.min():.2f}:1")

Class Distribution:
  0: 700 (70.0%)
  1: 300 (30.0%)

Imbalance Ratio: 2.33:1


In [4]:
# Convert to pandas for easier exploration
df = dataset.to_pandas()

print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (1000, 21)


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,target
0,0.0,6.0,4.0,3.0,1169.0,4.0,4.0,4.0,2.0,0.0,...,0.0,67.0,2.0,1.0,2.0,2.0,1.0,1.0,0.0,good
1,1.0,48.0,2.0,3.0,5951.0,0.0,2.0,2.0,1.0,0.0,...,0.0,22.0,2.0,1.0,1.0,2.0,1.0,0.0,0.0,bad
2,3.0,12.0,4.0,6.0,2096.0,0.0,3.0,2.0,2.0,0.0,...,0.0,49.0,2.0,1.0,1.0,1.0,2.0,0.0,0.0,good
3,0.0,42.0,2.0,2.0,7882.0,0.0,3.0,2.0,2.0,2.0,...,1.0,45.0,2.0,2.0,1.0,2.0,2.0,0.0,0.0,good
4,0.0,24.0,3.0,0.0,4870.0,0.0,2.0,3.0,2.0,0.0,...,3.0,53.0,2.0,2.0,2.0,2.0,2.0,0.0,0.0,bad


In [5]:
# Statistical summary of numerical features
print("Numerical Feature Statistics:")
display(df.describe().T)

Numerical Feature Statistics:


,count,mean,std,min,25%,50%,75%,max
checking_status,1000.0,1.577,1.257638,0.0,0.0,1.0,3.00,3.0
duration,1000.0,20.903,12.058814,4.0,12.0,18.0,24.00,72.0
credit_history,1000.0,2.545,1.083120,0.0,2.0,2.0,4.00,4.0
purpose,1000.0,2.828,2.744439,0.0,1.0,2.0,3.00,10.0
credit_amount,1000.0,3271.258,2822.736876,250.0,1365.5,2319.5,3972.25,18424.0
savings_status,1000.0,1.105,1.580023,0.0,0.0,0.0,2.00,4.0
employment,1000.0,2.384,1.208306,0.0,2.0,2.0,4.00,4.0
installment_commitment,1000.0,2.973,1.118715,1.0,2.0,3.0,4.00,4.0
personal_status,1000.0,1.682,0.708080,0.0,1.0,2.0,2.00,3.0
other_parties,1000.0,0.145,0.477706,0.0,0.0,0.0,0.00,2.0


## 3. Quick Baseline with One-Liner

Before investing time in feature engineering and tuning, let's establish a quick baseline using TuiML's high-level `train()` API. This single function call handles data loading, training, and evaluation.

In [6]:
import tuiml

# Quick baseline: Random Forest with 5-fold cross-validation
result = tuiml.train(
    algorithm="RandomForestClassifier",
    data="credit-g",
    target="class",
    cv=5
)

print("Baseline Results (RandomForestClassifier, 5-fold CV):")
print(f"  Metrics: {result.metrics}")

Baseline Results (RandomForestClassifier, 5-fold CV):
  Metrics: {'cv_accuracy_score_mean': 0.765, 'cv_accuracy_score_std': 0.03082207001484491, 'cv_f1_score_mean': 0.5264184702858744, 'cv_f1_score_std': 0.03381303801054501}


## 4. Algorithm Comparison

Credit scoring benefits from comparing multiple algorithm families. We use TuiML's `experiment()` function to systematically evaluate classifiers across different paradigms:
- **Tree-based:** Random Forest, C4.5 Decision Tree
- **Boosting:** XGBoost, AdaBoost
- **Probabilistic:** Naive Bayes
- **Instance-based:** K-Nearest Neighbors
- **Linear:** Logistic Regression

In [7]:
# Run experiment comparing 7 algorithms
exp = tuiml.experiment(
    algorithms=[
        "RandomForestClassifier",
        "XGBoostClassifier",
        "NaiveBayesClassifier",
        "C45TreeClassifier",
        "AdaBoostClassifier",
        "KNearestNeighborsClassifier",
        "LogisticRegression",
    ],
    datasets={"credit-g": (X, y)},
    cv=10,
    metrics=["accuracy", "f1_macro", "roc_auc"]
)

print("Experiment completed!")
print(exp.summary())

/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:11:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:12:00] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:12:01] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Experiment completed!
Experiment: Experiment
Validation: cross_validation
Metric: accuracy

Dataset: credit-g
--------------------------------------------------
  RandomForestClassifier: 0.7700 ± 0.0395
  XGBoostClassifier: 0.7590 ± 0.0416
  NaiveBayesClassifier: 0.7240 ± 0.0508
  C45TreeClassifier: 0.7000 ± 0.0000
  AdaBoostClassifier: 0.7370 ± 0.0249
  KNearestNeighborsClassifier: 0.6220 ± 0.0440
  LogisticRegression: 0.7640 ± 0.0429



In [8]:
# View detailed comparison table
print("=== Performance Matrix (Markdown) ===")
print(exp.to_markdown())

=== Performance Matrix (Markdown) ===
| Dataset | AdaBoostClassifier | C45TreeClassifier | KNearestNeighborsClassifier | LogisticRegression | NaiveBayesClassifier | RandomForestClassifier | XGBoostClassifier |
|---|---|---|---|---|---|---|---|
| credit-g | 0.7370 ± 0.0263 | 0.7000 ± 0.0000 ▼ | 0.6220 ± 0.0464 ▼ | 0.7640 ± 0.0453 | 0.7240 ± 0.0536 | **0.7700 ± 0.0416** ▲ | 0.7590 ± 0.0438 |
|---|---|---|---|---|---|---|---|
| **W/L/T** | 0/0/1 | 0/1/0 | 0/1/0 | 0/0/1 | 0/0/1 | 1/0/0 | 0/0/1 |


## 5. Workflow Approach

Now we build a proper ML pipeline using the `Workflow` builder. This fluent API chains preprocessing, training, and evaluation into a single readable pipeline.

For credit scoring, our pipeline:
1. **Impute** missing values with mean strategy
2. **Standardize** features (important for distance-based and gradient-based algorithms)
3. **Train** XGBoost with 100 estimators
4. **Cross-validate** with 10 folds using accuracy and F1

In [9]:
from tuiml.workflow import Workflow

# Build a complete pipeline with the Workflow builder
result = (
    Workflow("credit-g", target="class")
    .impute(strategy="mean")
    .standardize()
    .train("XGBoostClassifier", n_estimators=100)
    .cross_validate(cv=10, metrics=["accuracy_score", "f1_score"])
    .run()
)

print("Workflow Results (XGBoost Pipeline):")
print(f"  Metrics: {result.metrics}")
if result.cv_results:
    print(f"  CV Results: {result.cv_results}")

/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:12:08] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:12:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Workflow Results (XGBoost Pipeline):
  Metrics: {'cv_accuracy_score_mean': 0.768, 'cv_accuracy_score_std': 0.05362835071116768, 'cv_f1_score_mean': 0.5702267392587087, 'cv_f1_score_std': 0.1054926562447725}
  CV Results: {'scores': {'accuracy_score': [0.77, 0.83, 0.78, 0.82, 0.73, 0.67, 0.7, 0.81, 0.83, 0.74], 'f1_score': [0.6229508196721311, 0.6666666666666666, 0.5769230769230769, 0.6250000000000001, 0.49056603773584906, 0.47619047619047616, 0.31818181818181823, 0.6666666666666666, 0.6530612244897959, 0.606060606060606]}}


/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:12:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [10]:
# Try a second workflow with Random Forest and different settings
result_rf = (
    Workflow("credit-g", target="class")
    .impute(strategy="mean")
    .normalize(method="minmax")
    .resample(method="smote")
    .train("RandomForestClassifier", n_estimators=200, random_state=42)
    .cross_validate(cv=10, metrics=["accuracy_score", "f1_score"])
    .run()
)

print("Workflow Results (Random Forest + SMOTE):")
print(f"  Metrics: {result_rf.metrics}")

Workflow Results (Random Forest + SMOTE):
  Metrics: {'cv_accuracy_score_mean': 0.8435714285714285, 'cv_accuracy_score_std': 0.027930306851453073, 'cv_f1_score_mean': 0.8423580534684453, 'cv_f1_score_std': 0.02337808146845655}


## 6. Model Evaluation

Let's train the best-performing model on a holdout split and examine detailed evaluation metrics including the confusion matrix.

In [11]:
from tuiml.evaluation import train_test_split, accuracy_score, confusion_matrix
from tuiml.preprocessing.imputation import SimpleImputer
from tuiml.preprocessing.scaling import StandardScaler
from tuiml.algorithms.gradient_boosting import XGBoostClassifier

# Prepare data with preprocessing
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (800, 20)
Test set: (200, 20)


In [12]:
# Train the final model
model = XGBoostClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.2%}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(cm)
print(f"\nInterpretation:")
print(f"  True Negatives  (good predicted good): {cm[0][0]}")
print(f"  False Positives (good predicted bad):  {cm[0][1]}")
print(f"  False Negatives (bad predicted good):  {cm[1][0]}")
print(f"  True Positives  (bad predicted bad):   {cm[1][1]}")

Test Accuracy: 73.50%

Confusion Matrix:
[[119  21]
 [ 32  28]]

Interpretation:
  True Negatives  (good predicted good): 119
  False Positives (good predicted bad):  21
  False Negatives (bad predicted good):  32
  True Positives  (bad predicted bad):   28


/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [22:15:56] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [13]:
# Probability predictions for risk assessment
y_proba = model.predict_proba(X_test[:5])

print("Probability Predictions (first 5 applicants):")
print(f"{'Applicant':>10} {'P(good)':>10} {'P(bad)':>10} {'Predicted':>12} {'Actual':>10}")
print("-" * 55)
for i in range(5):
    pred_label = y_pred[i]
    actual_label = y_test[i]
    print(f"{'#' + str(i+1):>10} {y_proba[i][0]:>10.1%} {y_proba[i][1]:>10.1%} {pred_label:>12} {actual_label:>10}")

Probability Predictions (first 5 applicants):
 Applicant    P(good)     P(bad)    Predicted     Actual
-------------------------------------------------------
        #1     100.0%       0.0%            0          0
        #2     100.0%       0.0%            0          0
        #3      82.7%      17.3%            0          1
        #4      18.9%      81.1%            1          0
        #5      61.8%      38.2%            0          1


## 7. Save and Serve the Model

Once satisfied with performance, we save the trained model and deploy it as a REST API for real-time credit scoring.

In [14]:
import tuiml

# Save the model with metadata
tuiml.save(model, "credit_model.pkl", metadata={
    "dataset": "credit-g",
    "algorithm": "XGBoostClassifier",
    "accuracy": acc,
    "features": feature_names,
    "classes": ["good", "bad"]
})

print("Model saved as 'credit_model.pkl'")

Model saved as 'credit_model.pkl'


In [15]:
import time

# Serve the model as a REST API
info = tuiml.serve(result, port=8001, background=True, model_id="XGBoostClassifier")

# Wait for server to start
print("Starting server...")
time.sleep(2)

print(f"Server running at: {info['url']}")
print(f"API docs available at: {info['url']}/docs")

Starting server...
Server running at: http://127.0.0.1:8001
API docs available at: http://127.0.0.1:8001/docs


## 8. Making Predictions via API

Once the server is running, you can score new loan applicants using HTTP requests.

### Available Endpoints

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/models` | GET | List loaded models |
| `/models/{model_id}` | GET | Get model info |
| `/models/{model_id}/predict` | POST | Make predictions |
| `/models/{model_id}/predict_proba` | POST | Get class probabilities |
| `/health` | GET | Server health check |
| `/docs` | GET | Interactive API documentation (Swagger UI) |

### Using curl (Command Line)

```bash
# Check loaded models
curl http://127.0.0.1:8001/models

# Score a new loan applicant (20 features)
curl -X POST http://127.0.0.1:8001/models/XGBoostClassifier/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [[1, 6, 4, 2, 1169, 5, 5, 4, 3, 1, 4, 3, 67, 3, 2, 2, 3, 1, 2, 1]]}'

# Get probability scores for risk assessment
curl -X POST http://127.0.0.1:8001/models/XGBoostClassifier/predict_proba \
  -H "Content-Type: application/json" \
  -d '{"features": [[1, 6, 4, 2, 1169, 5, 5, 4, 3, 1, 4, 3, 67, 3, 2, 2, 3, 1, 2, 1]]}'
```

### Using Python requests

In [16]:
import requests

BASE_URL = "http://127.0.0.1:8001"
MODEL_ID = "XGBoostClassifier"

# Sample applicants for scoring
sample_data = {
    "features": [
        X_test[0].tolist(),  # First test applicant
        X_test[1].tolist(),  # Second test applicant
        X_test[2].tolist(),  # Third test applicant
    ]
}

# Make predictions
response = requests.post(
    f"{BASE_URL}/models/{MODEL_ID}/predict",
    json=sample_data
)
print("=== Credit Predictions ===")
if response.status_code == 200:
    print(response.json())
else:
    print(f"Error {response.status_code}:")
    print(response.json())

=== Credit Predictions ===
{'predictions': [0, 0, 1], 'model_id': 'XGBoostClassifier', 'model_class': 'XGBoostClassifier'}


In [17]:
# Get probability scores for risk stratification
response = requests.post(
    f"{BASE_URL}/models/{MODEL_ID}/predict_proba",
    json=sample_data
)

# Check for errors
if response.status_code != 200:
    print(f"Error: {response.status_code}")
    print(response.json())
else:
    result_api = response.json()
    
    print("=== Credit Risk Probabilities ===")
    # Handle case where 'classes' might not be present or is None
    classes = result_api.get('classes')
    if classes:
        print(f"Classes: {classes}\n")
    else:
        print("Classes: ['good', 'bad'] (inferred from dataset)\n")
    
    probabilities = result_api.get('probabilities', [])
    for i, probs in enumerate(probabilities):
        risk_level = "LOW" if probs[0] > 0.7 else "MEDIUM" if probs[0] > 0.5 else "HIGH"
        print(f"Applicant {i+1}: P(good)={probs[0]:.1%}, P(bad)={probs[1]:.1%} -> Risk: {risk_level}")

=== Credit Risk Probabilities ===
Classes: [0, 1]

Applicant 1: P(good)=99.1%, P(bad)=0.9% -> Risk: LOW
Applicant 2: P(good)=99.7%, P(bad)=0.3% -> Risk: LOW
Applicant 3: P(good)=7.2%, P(bad)=92.8% -> Risk: HIGH


## 9. Cleanup

Stop the running server and remove saved model artifacts.

In [18]:
import os

# Stop all running model servers
tuiml.stop_server()
print("Server stopped.")

# Remove saved model file
if os.path.exists("credit_model.pkl"):
    os.remove("credit_model.pkl")
    print("Removed credit_model.pkl")

Server stopped.
Removed credit_model.pkl


## 10. Summary and Key Takeaways

In this case study, we built a complete **credit scoring** pipeline using TuiML:

| Step | What We Did | TuiML API |
|------|-------------|-------------|
| Data Loading | Loaded the German Credit dataset | `load_dataset("credit")`, `get_dataset_info("credit")` |
| Exploration | Inspected features, class distribution, and statistics | `dataset.to_pandas()` |
| Quick Baseline | One-liner model with cross-validation | `tuiml.train("RandomForestClassifier", "credit", "class", cv=5)` |
| Comparison | Compared 7 algorithms on accuracy, F1, and AUC | `tuiml.experiment(algorithms, datasets, cv, metrics)` |
| Pipeline | Built a fluent preprocessing and training workflow | `Workflow("credit", target="class").impute().standardize().train().run()` |
| Evaluation | Analyzed accuracy, confusion matrix, and probability outputs | `accuracy_score()`, `confusion_matrix()`, `predict_proba()` |
| Deployment | Saved model and served as REST API | `tuiml.save()`, `tuiml.serve()` |
| Prediction | Called the API with curl and Python requests | HTTP POST to `/models/{id}/predict` |

### Key Takeaways

1. **Start simple.** The `tuiml.train()` one-liner establishes a quick baseline in a single line of code, providing immediate insight before deeper exploration.

2. **Compare algorithms systematically.** The `experiment()` API makes it easy to evaluate multiple classifier families. For credit scoring, ensemble methods (Random Forest, XGBoost) tend to outperform simpler models.

3. **Class imbalance matters.** With a 70/30 split between good and bad credit, techniques like SMOTE resampling can help the model learn better representations of the minority class.

4. **Probability outputs enable risk stratification.** Rather than a binary accept/reject, `predict_proba()` gives continuous risk scores that can be mapped to business tiers (LOW / MEDIUM / HIGH risk).

5. **Deployment is one line.** Going from a trained model to a production REST API requires just `tuiml.serve(result, port=8001, background=True)`, complete with auto-generated Swagger docs.